# Qwen3.8-Flash-Next on W7900D (Strata B0): quick test
The server runs one engine per card behind one OpenAI-compatible endpoint (port 8080). Start takes ~2 min.

In [ ]:
import json, os, time, urllib.request
BASE = 'http://127.0.0.1:8080'
KEY = os.environ.get('STRATA_API_KEY', '')
HDR = {'Authorization': f'Bearer {KEY}'} if KEY and KEY.lower() != 'none' else {}
def get(p):
    return json.load(urllib.request.urlopen(urllib.request.Request(BASE + p, headers=HDR), timeout=10))
for i in range(120):
    try:
        h = get('/health')
        if h.get('loaded'): break
    except Exception: pass
    time.sleep(5)
print(h)
print([m['id'] for m in get('/v1/models')['data']])

In [ ]:
body = {'model': get('/v1/models')['data'][0]['id'], 'max_tokens': 256,
        'messages': [{'role': 'user', 'content': 'Explain in three sentences what a mixture-of-experts model is.'}]}
req = urllib.request.Request(BASE + '/v1/chat/completions', data=json.dumps(body).encode(), headers={'Content-Type': 'application/json', **HDR})
t0 = time.time(); r = json.load(urllib.request.urlopen(req, timeout=600)); dt = time.time() - t0
print(r['choices'][0]['message']['content'])
print(f"{r['usage']['completion_tokens']} tokens in {dt:.1f} s ({r['usage']['completion_tokens']/dt:.1f} tok/s incl. prompt)")

In [ ]:
m = get('/metrics')
for rep in m.get('live', {}).get('replicas', []):
    print(rep)

**Using it from your own code (inside this instance):** any OpenAI client with `base_url='http://127.0.0.1:8080/v1'` and any `api_key` (no key is checked inside the instance).

**From the browser:** the same API is at `<this notebook's URL prefix>/proxy/8080/v1`, behind your Radeon Cloud login.

The endpoint is only reachable inside this instance or through that proxy; it is not exposed to the internet. Server log: `/tmp/b0-logs/autostart.log`.